In [1]:
from collections import defaultdict, Counter
import re
import math

class KneserNeyLanguageModel:

    def __init__(self, n=4, discount=0.75):
        self.n = n
        self.discount = discount

        self.ngram_counts = defaultdict(Counter)
        self.context_counts = Counter()

        self.continuation_count = Counter()
        self.predecessor_sets = defaultdict(set)

        self.vocab = set()

    # ----------------------------
    # Text preprocessing
    # ----------------------------
    def tokenize(self, text):

        text = text.lower()
        text = re.sub(r'[^a-z\s]', '', text)

        words = text.split()

        return words

    # ----------------------------
    # Training
    # ----------------------------
    def train(self, corpus):

        for sentence in corpus:

            tokens = ["<s>"]*(self.n-1) + self.tokenize(sentence) + ["</s>"]

            self.vocab.update(tokens)

            # Build counts
            for order in range(1, self.n+1):

                for i in range(len(tokens)-order+1):

                    ngram = tuple(tokens[i:i+order])

                    context = ngram[:-1]
                    word = ngram[-1]

                    self.ngram_counts[context][word] += 1
                    self.context_counts[context] += 1

                    if order == 2:
                        self.predecessor_sets[word].add(context)

        # Continuation counts
        for word in self.predecessor_sets:
            self.continuation_count[word] = len(self.predecessor_sets[word])

        self.total_continuations = sum(self.continuation_count.values())

    # ----------------------------
    # Recursive Kneser-Ney
    # ----------------------------
    def probability(self, word, context):

        if len(context) > self.n-1:
            context = context[-(self.n-1):]

        return self._recursive_prob(word, tuple(context))

    def _recursive_prob(self, word, context):

        # Base case (unigram)

        if len(context) == 0:

            return self.continuation_count[word] / self.total_continuations \
                   if self.total_continuations > 0 else 1/len(self.vocab)

        context_counter = self.ngram_counts.get(context, {})
        context_total = self.context_counts.get(context, 0)

        count = context_counter.get(word, 0)

        unique_follow = len(context_counter)

        lambda_weight = (
            self.discount * unique_follow / context_total
            if context_total > 0 else 1
        )

        lower_prob = self._recursive_prob(word, context[1:])

        if context_total == 0:
            return lower_prob

        return (
            max(count - self.discount, 0) / context_total
            + lambda_weight * lower_prob
        )

    # ----------------------------
    # Sentence probability
    # ----------------------------
    def sentence_probability(self, sentence):

        tokens = ["<s>"]*(self.n-1) + self.tokenize(sentence) + ["</s>"]

        prob = 1

        for i in range(self.n-1, len(tokens)):

            context = tokens[i-self.n+1:i]

            word = tokens[i]

            prob *= self.probability(word, context)

        return prob

    # ----------------------------
    # Log Probability
    # ----------------------------
    def sentence_log_probability(self, sentence):

        tokens = ["<s>"]*(self.n-1) + self.tokenize(sentence) + ["</s>"]

        logprob = 0

        for i in range(self.n-1, len(tokens)):

            context = tokens[i-self.n+1:i]

            word = tokens[i]

            p = self.probability(word, context)

            logprob += math.log(p)

        return logprob

    # ----------------------------
    # Perplexity
    # ----------------------------
    def perplexity(self, corpus):

        total_log = 0
        total_words = 0

        for sent in corpus:

            tokens = self.tokenize(sent)

            total_words += len(tokens)

            total_log += self.sentence_log_probability(sent)

        return math.exp(-total_log/total_words)

    # ----------------------------
    # Next word prediction
    # ----------------------------
    def predict_next(self, text, top_k=5):

        tokens = self.tokenize(text)

        context = tokens[-(self.n-1):]

        scores = {}

        for word in self.vocab:

            scores[word] = self.probability(word, context)

        return sorted(scores.items(),
                      key=lambda x:x[1],
                      reverse=True)[:top_k]


# ===================================================
# Example
# ===================================================

corpus = [

    "I love machine learning",

    "I love artificial intelligence",

    "Machine learning is amazing",

    "Artificial intelligence is powerful",

    "I enjoy learning python",

    "Python is useful for machine learning",

    "Language models predict words",

    "Machine learning models learn patterns"
]

model = KneserNeyLanguageModel(
            n=4,
            discount=0.75)

model.train(corpus)

print("Probability:")

print(model.probability(
        "learning",
        ["i","love","machine"]
))

print()

print("Sentence Probability:")

print(model.sentence_probability(
        "I love machine learning"
))

print()

print("Log Probability:")

print(model.sentence_log_probability(
        "I love machine learning"
))

print()

print("Perplexity:")

print(model.perplexity(corpus))

print()

print("Prediction:")

print(model.predict_next(
        "I love machine"
))

Probability:
0.9005580357142857

Sentence Probability:
0.04951359739082025

Log Probability:
-3.005507952365146

Perplexity:
2.8663210419534244

Prediction:
[('learning', 0.9005580357142857), ('</s>', 0.02109375), ('machine', 0.00904017857142857), ('is', 0.00904017857142857), ('python', 0.006026785714285714)]
